# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent, Path("yq/Desktop/muc")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据：{DATA_PATH}")
print(f"项目输出目录：{OUTPUT_DIR}")
print(f"原始数据形状：{raw_df.shape}")
raw_df.head()

原始数据：..\data\E Commerce Dataset.xlsx
项目输出目录：C:\Users\36816\Desktop\ecommerce-user-analysis-seed\output\day04_project
原始数据形状：(5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [2]:
# 在此写下你的答案：
# 1.每一行记录对应一位电商平台独立用户，完整存储该用户的注册时长、登录方式、城市等级、支付偏好、性别、APP 使用时长、下单品类、满意度、投诉、订单、返现等用户行为与属性信息。
# 2.Churn列是目标变量；其为用户流失标签，1表示已流失，0表示留存，本项目为用户流失预测建模任务
# 3.CustomerID 是用户唯一编号，数字大小不代表任何业务含义，不具备连续变量的大小、高低逻辑；仅用于区分每条样本、校验数据是否重复，属于分类标识字段。

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [3]:
def build_quality_report(data):
    dtype_series = data.dtypes
    miss_count = data.isna().sum()
    miss_pct = round((data.isna().mean() * 100), 2)
    unique_cnt = data.nunique()

    # 拼接成报告DataFrame
    report = pd.DataFrame({
        "字段类型": dtype_series,
        "缺失数量": miss_count,
        "缺失比例(%)": miss_pct,
        "唯一值数量": unique_cnt
    })
    # 行索引为字段名
    report.index.name = "字段名"
    return report
quality_before = build_quality_report(raw_df)
display(quality_before)

,字段类型,缺失数量,缺失比例(%),唯一值数量
字段名,,,,
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,object,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,object,0,0.00,7
Gender,object,0,0.00,2
HourSpendOnApp,float64,255,4.53,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [4]:
# TODO：完成项目初始审计
# 1. 整张表完全重复行数（不含第一条重复）
dup_all = raw_df.duplicated().sum()
print("完全重复行数：", dup_all)
# 2. CustomerID 重复记录条数
cid_dup = raw_df["CustomerID"].duplicated().sum()
print("CustomerID 重复记录数量：", cid_dup)
# 3. Churn 频数 + 流失率
churn_cnt = raw_df["Churn"].value_counts()
print("\nChurn 类别频数：")
print(churn_cnt)
churn_rate = raw_df["Churn"].mean()
print("流失率：", round(churn_rate, 4))
for col in ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]:
     print(f"\n{col}")
     print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复记录数量： 0

Churn 类别频数：
0    4682
1     948
Name: Churn, dtype: int64
流失率： 0.1684

PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: PreferredLoginDevice, dtype: int64

PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: PreferredPaymentMode, dtype: int64

PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: PreferedOrderCat, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [5]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [6]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。

    参数：
        data: 原始用户行为 DataFrame

    返回：
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # TODO：复制数据，避免覆盖原始数据
    # TODO：创建日志列表 logs
    # TODO：删除完全重复行，并记录日志
    # TODO：对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    # TODO：对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    # TODO：将 Churn 和 Complain 转为整数类型
    # TODO：返回 cleaned_df 与 cleaning_logdf = data.copy()
    df = data.copy()
    logs = []
    total_origin = df.shape[0]

    NUMERIC_MISSING_COLS = [
        "Tenure",
        "WarehouseToHome",
        "HourSpendOnApp",
        "OrderAmountHikeFromlastYear",
        "CouponUsed",
        "OrderCount",
        "DaySinceLastOrder",
    ]
    CATEGORY_MAPPINGS = {
        "PreferredLoginDevice": {"Phone": "Mobile Phone"},
        "PreferredPaymentMode": {"COD": "Cash on Delivery", "CC": "Credit Card"},
        "PreferedOrderCat": {"Mobile": "Mobile Phone"}
    }

    # 删除完全重复行
    dup_mask = df.duplicated()
    dup_num = dup_mask.sum()
    df = df[~dup_mask]
    logs.append({
        "处理步骤": "删除完全重复行",
        "处理规则": "删除所有全行完全重复的记录，保留首次出现样本",
        "处理前记录数": total_origin,
        "处理后记录数": df.shape[0],
        "影响记录数": dup_num
    })
    step1_after = df.shape[0]

    # 中位数填充数值缺失
    for col in NUMERIC_MISSING_COLS:
        before_miss = df[col].isna().sum()
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        after_miss = df[col].isna().sum()
        logs.append({
            "处理步骤": f"中位数填充字段 {col}",
            "处理规则": f"使用该字段中位数 {median_val:.2f} 填充缺失值",
            "处理前记录数": step1_after,
            "处理后记录数": df.shape[0],
            "影响记录数": before_miss - after_miss
        })

    # 类别同义标准化替换
    for col, replace_dict in CATEGORY_MAPPINGS.items():
        for old_val, new_val in replace_dict.items():
            affect_cnt = (df[col] == old_val).sum()
            df[col] = df[col].replace(old_val, new_val)
            logs.append({
                "处理步骤": f"类别标准化 {col}: {old_val}→{new_val}",
                "处理规则": f"统一同义分类，将简写/别名 {old_val} 替换为标准名称 {new_val}",
                "处理前记录数": df.shape[0],
                "处理后记录数": df.shape[0],
                "影响记录数": affect_cnt
            })

    # 强制类型转换
    int_cols = ["Churn", "Complain"]
    for col in int_cols:
        df[col] = df[col].astype(int)
        logs.append({
            "处理步骤": f"类型转换 {col} 为int",
            "处理规则": "将标签数值字段转为整数类型，统一数据格式",
            "处理前记录数": df.shape[0],
            "处理后记录数": df.shape[0],
            "影响记录数": 0
        })

    # 生成日志DataFrame
    cleaning_log = pd.DataFrame(logs)
    cleaned_df = df
    return cleaned_df, cleaning_log

### 任务 3：运行清洗函数并查看日志

In [7]:
# TODO：执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
display(cleaning_log)
cleaned_df.head()

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,删除所有全行完全重复的记录，保留首次出现样本,5630,5630,0
1,中位数填充字段 Tenure,使用该字段中位数 9.00 填充缺失值,5630,5630,264
2,中位数填充字段 WarehouseToHome,使用该字段中位数 14.00 填充缺失值,5630,5630,251
3,中位数填充字段 HourSpendOnApp,使用该字段中位数 3.00 填充缺失值,5630,5630,255
4,中位数填充字段 OrderAmountHikeFromlastYear,使用该字段中位数 15.00 填充缺失值,5630,5630,265
5,中位数填充字段 CouponUsed,使用该字段中位数 1.00 填充缺失值,5630,5630,256
6,中位数填充字段 OrderCount,使用该字段中位数 2.00 填充缺失值,5630,5630,258
7,中位数填充字段 DaySinceLastOrder,使用该字段中位数 3.00 填充缺失值,5630,5630,307
8,类别标准化 PreferredLoginDevice: Phone→Mobile Phone,统一同义分类，将简写/别名 Phone 替换为标准名称 Mobile Phone,5630,5630,1231
9,类别标准化 PreferredPaymentMode: COD→Cash on Delivery,统一同义分类，将简写/别名 COD 替换为标准名称 Cash on Delivery,5630,5630,365


字段名,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [8]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# TODO：构建 tenure_bins、tenure_labels，并用 pd.cut 新建 TenureGroup
tenure_bins = [-0.001, 6, 12, 24, 100]
tenure_labels = ["0-6个月", "7-12个月", "13-24个月", "24个月以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=tenure_bins, labels=tenure_labels, right=True)
# TODO：新建 IsMobileLogin，移动端为 1，其他设备为 0
cleaned_df["IsMobileLogin"] = (cleaned_df["PreferredLoginDevice"] == "Mobile Phone").astype(int)
# TODO：生成 outlier_report（每行对应一个待检查字段）
outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_result_list = []
for col in outlier_cols:
    res_dict = iqr_outlier_summary(cleaned_df[col])
    res_dict["字段名"] = col
    outlier_result_list.append(res_dict)

outlier_report = pd.DataFrame(outlier_result_list)
outlier_report = outlier_report[["字段名", "Q1", "Q3","下限", "上限", "候选异常值数量"]]
display(outlier_report)

,字段名,Q1,Q3,下限,上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [9]:
# TODO：完成业务规则检查
# business_rule_report = pd.DataFrame({
#     "规则": [...],
#     "不合规记录数": [...]
# })
rule_info = [
    ("使用时长小于 0", (cleaned_df["HourSpendOnApp"] < 0).sum()),
    ("仓库距离小于 0", (cleaned_df["WarehouseToHome"] < 0).sum()),
    ("订单数小于或等于 0", (cleaned_df["OrderCount"] <= 0).sum()),
    ("返现金额小于 0", (cleaned_df["CashbackAmount"] < 0).sum())
]

business_rule_report = pd.DataFrame({
    "规则": [item[0] for item in rule_info],
    "不合规记录数": [item[1] for item in rule_info]
})
display(business_rule_report)
"""处理结论：经业务规则校验，四条规则对应的不合规记录数量全部为0。
1. HourSpendOnApp、WarehouseToHome、CashbackAmount无负值，符合物理与业务逻辑；
2. OrderCount不存在小于等于0的异常记录，所有用户均存在下单行为;
3. 数据无违反业务逻辑脏数据,无需对该类记录进行删除或修正操作，可直接用于后续流失建模分析"""

,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0


'处理结论：经业务规则校验，四条规则对应的不合规记录数量全部为0。\n1. HourSpendOnApp、WarehouseToHome、CashbackAmount无负值，符合物理与业务逻辑；\n2. OrderCount不存在小于等于0的异常记录，所有用户均存在下单行为;\n3. 数据无违反业务逻辑脏数据,无需对该类记录进行删除或修正操作，可直接用于后续流失建模分析'

---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [10]:
quality_after = build_quality_report(cleaned_df)

NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder"
]

# 校验1：指定数值字段无缺失
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值字段仍存在缺失值"
# 校验2：分类同义值已全部标准化
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "登录设备存在未统一的Phone"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式存在未统一的COD"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式存在未统一的CC"
# 校验3：衍生特征已成功生成
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "缺少衍生字段TenureGroup/IsMobileLogin"
print("数据清洗验收全部通过")

# 4. 导出所有交付物（utf-8-sig编码，防止中文乱码）
# 清洗前质量报告
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
# 清洗后质量报告
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
# 清洗操作日志
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
# 最终清洗数据集
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
# 5. 打印两份审计报告
print("\n===== 候选异常值报告 =====")
display(outlier_report)
# 异常值报告导出
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False, encoding="utf-8-sig")
print("\n===== 业务规则违规报告 =====")
display(business_rule_report)
# 业务规则违规报告导出
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_report.csv", index=False, encoding="utf-8-sig")
# 6. 输出全部文件完整路径
print("\n===== 所有交付文件路径 =====")
file_list = [
    "data_quality_before.csv",
    "data_quality_after.csv",
    "cleaning_log.csv",
    "ecommerce_customer_cleaned.csv"
    "outlier\_report.csv",
    "business\_rule\_report.csv"
]
for file_name in file_list:
    full_path = OUTPUT_DIR / file_name
    print(full_path.resolve())

数据清洗验收全部通过

===== 候选异常值报告 =====


,字段名,Q1,Q3,下限,上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438



===== 业务规则违规报告 =====


,规则,不合规记录数
0,使用时长小于 0,0
1,仓库距离小于 0,0
2,订单数小于或等于 0,0
3,返现金额小于 0,0



===== 所有交付文件路径 =====
C:\Users\36816\Desktop\ecommerce-user-analysis-seed\output\day04_project\data_quality_before.csv
C:\Users\36816\Desktop\ecommerce-user-analysis-seed\output\day04_project\data_quality_after.csv
C:\Users\36816\Desktop\ecommerce-user-analysis-seed\output\day04_project\cleaning_log.csv
C:\Users\36816\Desktop\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csvoutlier\_report.csv
C:\Users\36816\Desktop\ecommerce-user-analysis-seed\output\day04_project\business\_rule\_report.csv


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？

  有7 个行为数值字段存在 4%-5.45% 缺失；登录设备、支付方式、商品品类存在同义别名；WarehouseToHome、OrderCount、CashbackAmount 存在 IQR 识别的候选异常；无重复用户与重复行。

2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？

  缺失值全局中位数填充；同义类别统一标准化；异常值仅统计记录，不自动删除.

3. 为什么清洗后的数据可以作为第五天分析的输入？

  已补全缺失、统一分类、新增分层衍生字段，数据规范完整，满足用户分层、流失建模分析需求

4. 哪些处理规则仍需要业务人员确认？

  Tenure 分层区间、IQR 异常样本是否真实业务错误需业务复核确认。

